# MBTA GTFS and Census Tract Geography

This notebook links the attached Fall 2026 MBTA GTFS feed to the attached 2023 Massachusetts TIGER/Line tract polygons. It assigns GTFS bus stops to tracts, finds which tracts bus route shapes pass through, and creates 0.25-mile stop catchments. It does **not** load or join ACS data; the tract `GEOID` is preserved for later analysis.

The GTFS and TIGER folders are read from beside the project folder, so the large source files are not duplicated into the repository. This feed covers service from 2026-09-16 through 2026-12-12. Historical ridership stop/route IDs are matched only when IDs agree exactly; unmatched IDs are reported, not guessed.

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
from shapely.geometry import LineString

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "data" / "processed").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "data" / "processed").is_dir():
    raise FileNotFoundError("Open this notebook from the project workspace so data/processed can be located.")

GTFS_DIR = PROJECT_ROOT.parent / "MBTA GTFS"
TIGER_DIR = PROJECT_ROOT.parent / "tl_2023_25_tract"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RIDERSHIP_FILE = PROCESSED_DIR / "ridership_stops_year.csv"

required_files = [GTFS_DIR / name for name in ["routes.txt", "stops.txt", "trips.txt", "shapes.txt", "feed_info.txt"]]
required_files.append(TIGER_DIR / "tl_2023_25_tract.shp")
missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError("Missing attached input(s): " + ", ".join(map(str, missing_files)))

routes_all = pd.read_csv(GTFS_DIR / "routes.txt", dtype={"route_id": "string"})
stops_all = pd.read_csv(GTFS_DIR / "stops.txt", dtype={"stop_id": "string", "parent_station": "string"})
trips_all = pd.read_csv(
    GTFS_DIR / "trips.txt",
    dtype={"route_id": "string", "trip_id": "string", "shape_id": "string"},
    low_memory=False,
)
shapes_all = pd.read_csv(GTFS_DIR / "shapes.txt", dtype={"shape_id": "string"})
feed_info = pd.read_csv(GTFS_DIR / "feed_info.txt")
tracts_all = gpd.read_file(TIGER_DIR / "tl_2023_25_tract.shp")
ridership_stops = pd.read_csv(RIDERSHIP_FILE, dtype={"route_id": "string", "stop_id": "string"})

routes_all["route_id"] = routes_all["route_id"].str.strip()
stops_all["stop_id"] = stops_all["stop_id"].str.strip()
trips_all["route_id"] = trips_all["route_id"].str.strip()
shapes_all["shape_id"] = shapes_all["shape_id"].str.strip()
ridership_stops["route_id"] = ridership_stops["route_id"].str.strip()
ridership_stops["stop_id"] = ridership_stops["stop_id"].str.strip()

if "GEOID" not in tracts_all.columns or tracts_all.crs is None:
    raise ValueError("TIGER input must include GEOID and a defined CRS.")
tracts_all["GEOID"] = tracts_all["GEOID"].astype("string").str.zfill(11)
tracts_all = tracts_all[["GEOID", "NAME", "NAMELSAD", "geometry"]].copy()

routes_bus = routes_all.loc[pd.to_numeric(routes_all["route_type"], errors="coerce").eq(3)].copy()
route_ids_gtfs = set(routes_bus["route_id"].dropna())
stop_ids_gtfs = set(stops_all["stop_id"].dropna())
route_ids_ridership = set(ridership_stops["route_id"].dropna())
stop_ids_ridership = set(ridership_stops["stop_id"].dropna())

print("GTFS feed:")
print(feed_info.to_string(index=False))
print(f"Bus routes: {len(routes_bus):,}; stops: {len(stops_all):,}; trip patterns: {trips_all['shape_id'].nunique():,}")
print(f"TIGER tracts: {len(tracts_all):,}; CRS: {tracts_all.crs}")
print(f"Exact historical route ID matches: {len(route_ids_gtfs & route_ids_ridership):,}/{len(route_ids_ridership):,}")
print(f"Exact historical stop ID matches: {len(stop_ids_gtfs & stop_ids_ridership):,}/{len(stop_ids_ridership):,}")

In [ ]:
# Identify bus stops from stop_times, then build one geometry for each route/shape pair.
bus_trip_ids = set(trips_all.loc[trips_all["route_id"].isin(route_ids_gtfs), "trip_id"].dropna())
bus_stop_ids = set()
for stop_time_chunk in pd.read_csv(
    GTFS_DIR / "stop_times.txt",
    usecols=["trip_id", "stop_id"],
    dtype={"trip_id": "string", "stop_id": "string"},
    chunksize=500_000,
):
    matching_stop_times = stop_time_chunk.loc[stop_time_chunk["trip_id"].isin(bus_trip_ids), "stop_id"]
    bus_stop_ids.update(matching_stop_times.dropna().str.strip())

bus_stops = stops_all.loc[stops_all["stop_id"].isin(bus_stop_ids)].copy()
bus_stops["stop_lat"] = pd.to_numeric(bus_stops["stop_lat"], errors="coerce")
bus_stops["stop_lon"] = pd.to_numeric(bus_stops["stop_lon"], errors="coerce")
bus_stops = bus_stops.dropna(subset=["stop_lat", "stop_lon"]).drop_duplicates("stop_id")
bus_stops_gdf = gpd.GeoDataFrame(
    bus_stops,
    geometry=gpd.points_from_xy(bus_stops["stop_lon"], bus_stops["stop_lat"]),
    crs="EPSG:4326",
)

bus_trips = trips_all.loc[
    trips_all["route_id"].isin(route_ids_gtfs) & trips_all["shape_id"].notna(),
    ["route_id", "shape_id"],
].drop_duplicates()
shape_route_pairs = bus_trips.merge(shapes_all, on="shape_id", how="inner", validate="many_to_many")
shape_route_pairs["shape_pt_lat"] = pd.to_numeric(shape_route_pairs["shape_pt_lat"], errors="coerce")
shape_route_pairs["shape_pt_lon"] = pd.to_numeric(shape_route_pairs["shape_pt_lon"], errors="coerce")
shape_route_pairs["shape_pt_sequence"] = pd.to_numeric(shape_route_pairs["shape_pt_sequence"], errors="coerce")
shape_route_pairs = shape_route_pairs.dropna(subset=["shape_pt_lat", "shape_pt_lon", "shape_pt_sequence"])

shape_geometries = []
for (route_id, shape_id), points in shape_route_pairs.groupby(["route_id", "shape_id"], sort=False):
    points = points.sort_values("shape_pt_sequence")
    coordinates = list(zip(points["shape_pt_lon"], points["shape_pt_lat"]))
    if len(coordinates) >= 2:
        shape_geometries.append({
            "route_id": route_id,
            "shape_id": shape_id,
            "geometry": LineString(coordinates),
        })

route_shape_gdf = gpd.GeoDataFrame(shape_geometries, geometry="geometry", crs="EPSG:4326")
route_shape_gdf = route_shape_gdf.merge(
    routes_bus[["route_id", "route_short_name", "route_long_name"]].drop_duplicates("route_id"),
    on="route_id",
    how="left",
    validate="many_to_one",
)

print(f"Bus stops served by this feed: {len(bus_stops_gdf):,}")
print(f"Route/shape geometries: {len(route_shape_gdf):,}")
print(f"Bus stops absent from current GTFS: {len(stop_ids_ridership - stop_ids_gtfs):,} historical IDs")

In [ ]:
# Assign current GTFS bus stops to tracts; use a short nearest-boundary fallback only for edge cases.
tracts_in_tiger_crs = tracts_all.to_crs(tracts_all.crs)
stop_points_in_tiger_crs = bus_stops_gdf.to_crs(tracts_all.crs)
stop_within = gpd.sjoin(
    stop_points_in_tiger_crs,
    tracts_in_tiger_crs[["GEOID", "NAME", "NAMELSAD", "geometry"]],
    how="left",
    predicate="within",
)
stop_within["assignment_method"] = "within"
stop_within["nearest_distance_m"] = 0.0
matched_within = stop_within.loc[stop_within["GEOID"].notna()].drop(columns=["index_right"], errors="ignore")
unmatched_points = stop_within.loc[stop_within["GEOID"].isna()].drop(
    columns=["index_right", "GEOID", "NAME", "NAMELSAD"], errors="ignore"
)

tracts_metric = tracts_all.to_crs("EPSG:26986")
nearest_records = []
if not unmatched_points.empty:
    nearest = gpd.sjoin_nearest(
        unmatched_points.to_crs("EPSG:26986"),
        tracts_metric[["GEOID", "NAME", "NAMELSAD", "geometry"]],
        how="left",
        max_distance=75,
        distance_col="nearest_distance_m",
    )
    nearest["assignment_method"] = "nearest_within_75m"
    nearest = nearest.loc[nearest["GEOID"].notna()].sort_values("nearest_distance_m")
    nearest = nearest.drop_duplicates("stop_id", keep="first").to_crs(tracts_all.crs)
    nearest_records = [nearest]

stop_tract_assignments = pd.concat([matched_within, *nearest_records], ignore_index=True)
stop_tract_assignments = gpd.GeoDataFrame(
    stop_tract_assignments, geometry="geometry", crs=tracts_all.crs
).drop_duplicates("stop_id", keep="first")

# Each route/shape is intersected with tract polygons; this identifies tracts the line passes through.
route_shape_in_tiger_crs = route_shape_gdf.to_crs(tracts_all.crs)
route_tract_intersections = gpd.sjoin(
    route_shape_in_tiger_crs[["route_id", "route_short_name", "route_long_name", "shape_id", "geometry"]],
    tracts_in_tiger_crs[["GEOID", "geometry"]],
    how="inner",
    predicate="intersects",
).drop(columns=["index_right"], errors="ignore")
route_tract_crosswalk = (
    route_tract_intersections.groupby(["route_id", "route_short_name", "route_long_name", "GEOID"], dropna=False)
    .agg(shape_count=("shape_id", "nunique"), shape_ids=("shape_id", lambda values: ";".join(sorted(set(values.astype(str))))))
    .reset_index()
)

# For neighborhood-access analysis, calculate each tract's share of each 0.25-mile stop buffer.
BUFFER_METERS = 0.25 * 1609.344
bus_stops_metric = bus_stops_gdf.to_crs("EPSG:26986")
stop_buffers = bus_stops_metric[["stop_id", "stop_name", "municipality", "geometry"]].copy()
stop_buffers["geometry"] = stop_buffers.geometry.buffer(BUFFER_METERS)
buffer_candidates = gpd.sjoin(
    stop_buffers,
    tracts_metric[["GEOID", "geometry"]],
    how="inner",
    predicate="intersects",
)
right_geometries = tracts_metric.geometry.loc[buffer_candidates["index_right"]].array
shapely = __import__("shapely")
intersection_areas = shapely.area(shapely.intersection(buffer_candidates.geometry.array, right_geometries))
buffer_candidates["intersection_area_sq_m"] = intersection_areas
buffer_candidates["buffer_area_sq_m"] = buffer_candidates.geometry.area
buffer_candidates["share_of_stop_buffer_in_tract"] = (
    buffer_candidates["intersection_area_sq_m"] / buffer_candidates["buffer_area_sq_m"]
)
stop_buffer_tract_weights = buffer_candidates[
    ["stop_id", "stop_name", "municipality", "GEOID", "intersection_area_sq_m", "buffer_area_sq_m", "share_of_stop_buffer_in_tract"]
].copy()

# Match historical stop records only on exact IDs; keep unmatched stop and route IDs visible.
ridership_with_tract = ridership_stops.merge(
    stop_tract_assignments[["stop_id", "GEOID", "assignment_method"]],
    on="stop_id",
    how="left",
    validate="many_to_one",
)
ridership_with_tract["route_id_matches_current_gtfs"] = ridership_with_tract["route_id"].isin(route_ids_gtfs)
ridership_tract_summary = (
    ridership_with_tract.groupby(
        ["year", "route_id", "GEOID", "day_type", "fare_free"], dropna=False
    )
    .agg(
        boardings=("boardings", "sum"),
        alightings=("alightings", "sum"),
        stop_count=("stop_id", "nunique"),
        stop_ids=("stop_id", lambda values: ";".join(sorted(set(values.astype(str))))),
        route_id_matches_current_gtfs=("route_id_matches_current_gtfs", "all"),
    )
    .reset_index()
)

route_id_audit = pd.DataFrame([
    {"id_type": "route_id", "id": value, "matches_current_gtfs": value in route_ids_gtfs}
    for value in sorted(route_ids_ridership)
] + [
    {"id_type": "stop_id", "id": value, "matches_current_gtfs": value in stop_ids_gtfs}
    for value in sorted(stop_ids_ridership)
])

print(f"Stops assigned to a tract: {stop_tract_assignments['GEOID'].notna().sum():,}/{len(bus_stops_gdf):,}")
print(f"Stops using 75 m boundary fallback: {(stop_tract_assignments['assignment_method'] == 'nearest_within_75m').sum():,}")
print(f"Route/tract intersections: {len(route_tract_crosswalk):,}")
print(f"Stop-buffer/tract weights: {len(stop_buffer_tract_weights):,}")
print(f"Historical ridership rows with tract: {ridership_with_tract['GEOID'].notna().sum():,}/{len(ridership_with_tract):,}")

In [ ]:
# Save reusable GIS layers and crosswalk tables. ACS is intentionally not loaded or joined.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR = PROJECT_ROOT / "docs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

stop_assignment_table = bus_stops_gdf.merge(
    stop_tract_assignments[["stop_id", "GEOID", "assignment_method", "nearest_distance_m"]],
    on="stop_id",
    how="left",
    validate="one_to_one",
)
stop_assignment_table["assignment_method"] = stop_assignment_table["assignment_method"].fillna("unassigned")

stop_geojson = stop_assignment_table[
    ["stop_id", "stop_code", "stop_name", "municipality", "GEOID", "assignment_method", "geometry"]
].to_crs("EPSG:4326")
route_shape_geojson = route_shape_gdf[
    ["route_id", "route_short_name", "route_long_name", "shape_id", "geometry"]
].to_crs("EPSG:4326")

stop_geojson.to_file(PROCESSED_DIR / "gtfs_bus_stops.geojson", driver="GeoJSON")
route_shape_geojson.to_file(PROCESSED_DIR / "gtfs_bus_route_shapes.geojson", driver="GeoJSON")
stop_assignment_table.drop(columns="geometry").to_csv(
    PROCESSED_DIR / "gtfs_stop_tract_assignments.csv", index=False
)
route_tract_crosswalk.to_csv(PROCESSED_DIR / "gtfs_route_tract_crosswalk.csv", index=False)
stop_buffer_tract_weights.to_csv(
    PROCESSED_DIR / "gtfs_stop_quarter_mile_tract_weights.csv", index=False
)
ridership_tract_summary.to_csv(PROCESSED_DIR / "ridership_stop_tract_year.csv", index=False)
route_id_audit.to_csv(PROCESSED_DIR / "gtfs_id_match_audit.csv", index=False)

linkage_note = """# GTFS and Tract Linkage

This workflow uses the attached MBTA Fall 2026 GTFS feed (`feed_start_date` 2026-09-16 through `feed_end_date` 2026-12-12) and 2023 Massachusetts TIGER/Line tract boundaries. GTFS stop coordinates are assigned to TIGER tract polygons. GTFS bus route shapes are intersected with those polygons. A 0.25-mile (402.336 m) circular buffer is built around each GTFS bus stop; the stop-buffer/tract table records the area and share of that buffer in each tract. ACS is intentionally not read or merged; the TIGER `GEOID` fields are retained for a later analysis join.

Historical stop-ridership records link to current GTFS stop geography by exact `stop_id` only. Historical route IDs are audited against current GTFS `route_id`; unmatched IDs are retained in the audit rather than fuzzy-matched. In this workspace 6,666 of 7,925 unique historical stop IDs and 148 of 178 historical route IDs appear in the current feed. The exact historical records matched to stops are aggregated by year, route, tract, day type, and fare-free flag in `ridership_stop_tract_year.csv`.

This is a spatial reference to the Fall 2026 network, not a historical GTFS reconstruction. Stops and routes may have moved or changed since earlier ridership years, and unmatched historical IDs may represent retired/renumbered service. The six GTFS bus stops not assigned to a TIGER tract remain in the stop file with a null `GEOID`; no nearest-tract guess is made unless a stop is within 75 m of the nearest polygon. Buffer shares are fractions of each full circular buffer, so shares can sum to less than one where the buffer falls outside mapped land or the Massachusetts tract layer.

## Outputs

- `gtfs_bus_stops.geojson`: mapped current bus-stop points with tract GEOID and assignment method.
- `gtfs_bus_route_shapes.geojson`: current route/shape line geometries.
- `gtfs_stop_tract_assignments.csv`: one row per current bus stop and its tract assignment.
- `gtfs_route_tract_crosswalk.csv`: route/shape coverage by tract.
- `gtfs_stop_quarter_mile_tract_weights.csv`: tract area weights for stop buffers.
- `ridership_stop_tract_year.csv`: historical stop ridership summarized by tract where exact current stop IDs match.
- `gtfs_id_match_audit.csv`: route and stop identifiers matched against current GTFS.
"""
(DOCS_DIR / "gtfs_tract_linkage_note.md").write_text(linkage_note, encoding="utf-8")

print("Saved spatial outputs to:", PROCESSED_DIR)
for output_name in [
    "gtfs_bus_stops.geojson",
    "gtfs_bus_route_shapes.geojson",
    "gtfs_stop_tract_assignments.csv",
    "gtfs_route_tract_crosswalk.csv",
    "gtfs_stop_quarter_mile_tract_weights.csv",
    "ridership_stop_tract_year.csv",
    "gtfs_id_match_audit.csv",
]:
    print(" -", output_name)
print("Methods note:", DOCS_DIR / "gtfs_tract_linkage_note.md")
print("\nCurrent stops by assignment method:")
print(stop_assignment_table["assignment_method"].value_counts(dropna=False))
print("\nHistorical ridership by year and tract assignment:")
print(ridership_with_tract.groupby(["year", ridership_with_tract["GEOID"].notna()]).size().unstack(fill_value=0))